[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/cdg-ime/blob/main/Prospecção.ipynb)

# Ciência de Dados Geoespaciais - Prospecção

**Maj Diego - 2° Semestre / 2026**

**Objetivos**

1. Conceituar ciência de dados;
2. Especificar dados para análise;
3. Identificar datasets públicos;
4. Conceituar API e como ser cliente de uma API;
5. Como efetuar requisições para uma API;
6. Como obter dados sem serviço de API;

**Referência:**
- LONGLEY, P.A. et al. *Geographic Information Science and Systems*. Wiley, 2015.
- MCKINNEY, W. *Python for Data Analysis*. O'Reilly, 2022.
- REY, S.J.; ARRIBAS-BEL, D.; WOLF, L.J. *Geographic Data Science with Python*. CRC Press, 2023.

## 1. Conceituar ciência de dados

**Ciência de Dados** é um campo interdisciplinar que combina estatística, computação e conhecimento de domínio para extrair conhecimento e insights a partir de dados estruturados e não estruturados.

### 1.1 O que é Ciência de Dados Geoespaciais?

A **Ciência de Dados Geoespaciais** aplica os métodos de ciência de dados a dados que possuem **componente espacial** — ou seja, dados que podem ser referenciados a um local na superfície terrestre.

Exemplos de perguntas que ela responde:
- Onde há maior risco de deslizamento de encosta?
- Como a temperatura urbana varia em função da cobertura vegetal?
- Quais regiões apresentam maior vulnerabilidade logística?

### 1.2 O Pipeline de Ciência de Dados

| Etapa | Descrição | Ferramentas comuns |
|-------|-----------|--------------------|
| **Prospecção** | Identificar e coletar dados | APIs, web scraping, portais abertos |
| **Tratamento/Pré-processamento** | Limpar, transformar e integrar | pandas, geopandas, GDAL |
| **Exploração/Mineração** | Analisar estatísticas e visualizar | matplotlib, seaborn, folium |
| **Modelagem** | Aplicar algoritmos e modelos | scikit-learn, statsmodels |
| **Análise do resultado** | Apresentar resultados | dashboards, mapas, relatórios |

### 1.3 Primeiro de Tudo: Defina a Pergunta

Antes de coletar qualquer dado, é essencial formular uma **pergunta clara e mensurável**. Perguntas vagas levam a coletas desnecessárias e análises sem foco.

| Pergunta vaga | Pergunta bem formulada |
|--------------|------------------------|
| "Onde há crime?" | "Qual a densidade de ocorrências de roubo por km² nos bairros do Rio de Janeiro em 2023?" |
| "Como está o trânsito?" | "Qual o tempo médio de deslocamento entre Niterói e o Centro do Rio às 8h nos dias úteis?" |

## 2. Especificar dados para análise

### 2.1 Tipos de Dados quanto à Estrutura

| Tipo | Descrição | Exemplo |
|------|-----------|----------|
| **Estruturado** | Organizado em tabelas com schema fixo | Censo IBGE em .csv |
| **Semi-estruturado** | Tem organização, mas sem schema rígido | JSON de API, XML |
| **Não estruturado** | Sem organização predefinida | Texto livre, imagens de satélite |

### 2.2 Tipos de Dados Geoespaciais

#### Modelo Vetorial
Representa feições geográficas como geometrias discretas:

- **Ponto** (`Point`): cidades, estações, ocorrências
- **Linha** (`LineString`): rios, rodovias, redes de transporte
- **Polígono** (`Polygon`): municípios, bacias hidrográficas, lotes

```
  Ponto          Linha          Polígono
    •           /\             ╔════╗
               /  \            ║    ║
              /    \           ╚════╝
```

In [ ]:
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt

# ── Geometrias em WKT ────────────────────────────────────────
ponto   = wkt.loads("POINT (0 0)")
linha   = wkt.loads("LINESTRING (2 -1, 2.5 0, 3 1, 3.5 0, 4 -1)")
poligono = wkt.loads("POLYGON ((6 -1, 8 -1, 8 1, 6 1, 6 -1))")

# ── GeoDataFrame com os três tipos ───────────────────────────
gdf = gpd.GeoDataFrame(
    {
        "tipo"  : ["Ponto", "Linha", "Polígono"],
        "wkt"   : [ponto.wkt, linha.wkt, poligono.wkt],
        "geometry": [ponto, linha, poligono],
    },
    crs="EPSG:4326",
)

# ── Plot ──────────────────────────────────────────────────────
cores = ["#e74c3c", "#2980b9", "#27ae60"]

fig, ax = plt.subplots(figsize=(9, 4))

# Polígono (fundo + borda)
gdf[gdf.geom_type == "Polygon"].plot(
    ax=ax, color="#27ae60", edgecolor="#1a6e3c", linewidth=2, alpha=0.6, label="Polígono"
)
# Linha
gdf[gdf.geom_type == "LineString"].plot(
    ax=ax, color="#2980b9", linewidth=2.5, label="Linha"
)
# Ponto
gdf[gdf.geom_type == "Point"].plot(
    ax=ax, color="#e74c3c", markersize=120, zorder=5, label="Ponto"
)

# Rótulos de tipo sobre cada geometria
for _, row in gdf.iterrows():
    cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
    ax.annotate(row["tipo"], xy=(cx, cy), xytext=(0, 14),
                textcoords="offset points", ha="center",
                fontsize=11, fontweight="bold")

ax.set_title("Geometrias WKT com GeoPandas", fontsize=13)
ax.legend(loc="upper left")
ax.set_axis_off()
plt.tight_layout()
plt.savefig("geometrias_wkt.png", dpi=150)
plt.show()

# ── Inspecionar WKTs ─────────────────────────────────────────
print(gdf[["tipo", "wkt"]].to_string(index=False))

#### Modelo Raster
Representa dados como uma grade regular de células (pixels), cada uma com um valor:
- Imagens de satélite (Landsat, Sentinel)
- Modelos Digitais de Elevação (MDE)
- Grades climáticas (temperatura, precipitação)



### 2.3 Atributos Essenciais de um Dado Geoespacial

Ao especificar um dado, defina:

1. **Extensão espacial** – qual área geográfica? (Brasil, município, bairro)
2. **Resolução espacial** – qual o menor detalhe representado? (1m, 30m, município)
3. **Resolução temporal** – com que frequência foi coletado? (diário, anual, único)
4. **Sistema de Referência de Coordenadas (SRC/CRS)** – em qual projeção? (WGS84, SIRGAS2000)
5. **Formato** – Shapefile, GeoJSON, GeoTIFF, CSV com lat/lon?
6. **Licença** – pode ser usado livremente? Requer atribuição?

### 2.4 Exercício de Especificação

Para o problema apresentado (crescimento populacional × rodovias), preencha a tabela:

| Dado necessário | Fonte provável | Formato | Resolução | CRS |
|----------------|---------------|---------|-----------|-----|
| Limites municipais | IBGE | Shapefile | Município | SIRGAS2000 |
| População 2010/2022 | IBGE Censo | CSV/API | Município | — |
| Rodovias federais | DNIT / OSM | Shapefile / GeoJSON | 1:250.000 | WGS84 |

## O Problema da prospecção de dados geoespaciais

Imagine que você precisa responder à seguinte pergunta:

> **"Quais municípios brasileiros apresentaram crescimento populacional acima da média entre 2010 e 2022, e como esse crescimento se relaciona com a proximidade de rodovias federais?"**

Para responder a essa pergunta, você precisará:
- Dados populacionais por município (IBGE)
- Geometria dos municípios (shapefiles)
- Localização de rodovias federais (DNIT/OpenStreetMap)


## 3. Identificar datasets públicos

Há uma vasta quantidade de dados geoespaciais disponíveis publicamente. Conhecer os principais repositórios é fundamental.

### 3.1 Portais Brasileiros

| Portal | URL | Conteúdo |
|--------|-----|----------|
| **IBGE** | [geoftp.ibge.gov.br](https://geoftp.ibge.gov.br) | Malhas municipais, censo, estatísticas |
| **IBGE API** | [servicodados.ibge.gov.br](https://servicodados.ibge.gov.br) | Dados censitários via API REST |
| **INPE** | [queimadas.dgi.inpe.br](http://queimadas.dgi.inpe.br) | Focos de queimadas, imagens |
| **ANA** | [hidroweb.ana.gov.br](https://hidroweb.ana.gov.br) | Hidrologia, postos fluviométricos |
| **ANTT/DNIT** | [dados.gov.br](https://dados.gov.br) | Rodovias, ferrovias |
| **dados.gov.br** | [dados.gov.br](https://dados.gov.br) | Portal central de dados abertos do governo |

### 3.2 Portais Internacionais

| Portal | Conteúdo |
|--------|----------|
| **OpenStreetMap** | Mapa colaborativo global (ruas, edificações, POIs) |
| **NASA EarthData** | Imagens de satélite, clima, oceanos |
| **Copernicus (ESA)** | Sentinel, dados ambientais, Europa e global |
| **Natural Earth** | Dados vetoriais globais para cartografia |
| **GADM** | Divisões administrativas globais |
| **Our World in Data** | Dados socioeconômicos globais com API |

### 3.3 Exemplo: Baixando Malha Municipal do IBGE

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Leitura direta da malha municipal do IBGE (2022) via URL
url_ibge = (
    "https://geoftp.ibge.gov.br/organizacao_do_territorio/"
    "malhas_territoriais/malhas_municipais/municipio_2022/"
    "Brasil/BR/BR_Municipios_2022.zip"
)

# Leitura com geopandas (suporta leitura direta de .zip)
municipios = gpd.read_file(url_ibge)

print(f"Total de municípios: {len(municipios)}")
print(f"CRS: {municipios.crs}")
print(municipios.head(3))

municipios.plot(figsize=(10, 8), edgecolor='gray', linewidth=0.3)
plt.title("Municípios do Brasil - IBGE 2022")
plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Conceituar API e como ser cliente de uma API

### 4.1 O que é uma API?

**API** (*Application Programming Interface*) é um conjunto de regras e protocolos que permite que dois sistemas de software se comuniquem. Em ciência de dados, usamos principalmente **APIs Web** baseadas no protocolo HTTP.

```
         Requisição HTTP (GET /dados?municipio=3304557)
Cliente ────────────────────────────────────────────────▶ Servidor (API)
        ◀────────────────────────────────────────────────
              Resposta HTTP 200 OK  {"populacao": 515317}
```

### 4.2 Anatomia de uma URL de API

```
https://servicodados.ibge.gov.br/api/v1/localidades/municipios/3304557
│──────────────────────────────────────│──────────────│──────────────│
         Base URL                        Endpoint        Parâmetro
                                        (recurso)       (código IBGE)
```

### 4.3 Métodos HTTP mais comuns

| Método | Uso | Analogia |
|--------|-----|----------|
| `GET` | Buscar/ler dados | Consultar uma ficha |
| `POST` | Enviar/criar dados | Preencher um formulário |
| `PUT` | Atualizar dados | Substituir uma ficha |
| `DELETE` | Remover dados | Apagar um registro |

> Em **ciência de dados**, usamos quase exclusivamente `GET` — apenas queremos *ler* dados.

### 4.4 Códigos de Status HTTP

| Código | Significado |
|--------|-------------|
| `200 OK` | Sucesso |
| `400 Bad Request` | Parâmetro inválido na requisição |
| `401 Unauthorized` | Autenticação necessária (API Key) |
| `403 Forbidden` | Sem permissão de acesso |
| `404 Not Found` | Recurso não encontrado |
| `429 Too Many Requests` | Limite de requisições atingido |
| `500 Internal Server Error` | Erro no servidor |

### 4.5 Formatos de Resposta

A maioria das APIs retorna dados em **JSON** (*JavaScript Object Notation*):

```json
{
  "id": 3304557,
  "nome": "Niterói",
  "microrregiao": {
    "id": 33009,
    "nome": "Rio de Janeiro"
  }
}
```

APIs geoespaciais frequentemente retornam **GeoJSON**:

```json
{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [-43.1033, -22.8832]
  },
  "properties": {
    "nome": "Niterói",
    "populacao": 515317
  }
}
```

### 4.6 Autenticação

Muitas APIs exigem uma **chave de acesso (API Key)** para controle de uso:

```python
# Exemplos de autenticação

# 1. Via parâmetro na URL
url = "https://api.exemplo.com/dados?api_key=SUA_CHAVE"

# 2. Via cabeçalho (header) HTTP
headers = {"Authorization": "Bearer SUA_CHAVE"}
response = requests.get(url, headers=headers)

# Boas práticas: NUNCA coloque a chave diretamente no código!
import os
api_key = os.environ.get("MINHA_API_KEY")  # ✅ Correto
api_key = "abc123xyz"                       # ❌ Evitar
```

## 5. Como efetuar requisições para uma API

Em Python, a biblioteca padrão para fazer requisições HTTP é a `requests`.

### 5.1 Instalação e importação

In [ ]:
# A maioria dos ambientes já possui a biblioteca instalada
# !pip install requests

import requests
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

### 5.2 Requisição básica — API do IBGE (Localidades)

In [ ]:
# Endpoint da API de Localidades do IBGE
url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios/3304557"

# Fazendo a requisição GET
resposta = requests.get(url)

# Verificando o status
print(f"Status HTTP: {resposta.status_code}")

# Parseando o JSON retornado
dados = resposta.json()
print(json.dumps(dados, indent=2, ensure_ascii=False))

### 5.3 Requisição com parâmetros — Censo 2022 (população por UF)

In [ ]:
# API do IBGE - Agregados (SIDRA)
# Tabela 9514: População residente por município (Censo 2022)
# Documentação: https://apisidra.ibge.gov.br/

url_sidra = "https://apisidra.ibge.gov.br/values/t/9514/n6/all/v/93/p/2022/d/v93 0"

resposta = requests.get(url_sidra)
dados = resposta.json()

# Os dados vêm como lista de dicionários; o primeiro item é o cabeçalho
df_pop = pd.DataFrame(dados[1:])  # Pula o header da API
df_pop = df_pop[['D1C', 'D1N', 'V']].rename(columns={
    'D1C': 'codigo_ibge',
    'D1N': 'municipio',
    'V': 'populacao_2022'
})
df_pop['populacao_2022'] = pd.to_numeric(df_pop['populacao_2022'], errors='coerce')

print(f"Municípios obtidos: {len(df_pop)}")
print(df_pop.head())
print(f"\nPopulação total: {df_pop['populacao_2022'].sum():,.0f}")

### 5.4 Exemplo avançado — Overpass API (OpenStreetMap)

A **Overpass API** permite consultar dados do OpenStreetMap com uma linguagem de query própria.

In [ ]:
import requests
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

# Consulta Overpass: hospitais em Niterói-RJ
overpass_url = "http://overpass-api.de/api/interpreter"

# Overpass QL (Query Language)
query = """
[out:json][timeout:25];
area["name"="Niterói"]["admin_level"="8"]->.searchArea;
(
  node["amenity"="hospital"](area.searchArea);
  way["amenity"="hospital"](area.searchArea);
);
out center;
"""

resposta = requests.get(overpass_url, params={'data': query})
dados = resposta.json()

# Extraindo coordenadas dos resultados
hospitais = []
for elemento in dados['elements']:
    lat = elemento.get('lat') or elemento.get('center', {}).get('lat')
    lon = elemento.get('lon') or elemento.get('center', {}).get('lon')
    nome = elemento.get('tags', {}).get('name', 'Sem nome')
    if lat and lon:
        hospitais.append({'nome': nome, 'lat': lat, 'lon': lon})

df_hosp = pd.DataFrame(hospitais)
gdf_hosp = gpd.GeoDataFrame(
    df_hosp,
    geometry=[Point(row['lon'], row['lat']) for _, row in df_hosp.iterrows()],
    crs='EPSG:4326'
)

print(f"Hospitais encontrados: {len(gdf_hosp)}")
print(gdf_hosp[['nome', 'lat', 'lon']].to_string())

# Mapa simples
gdf_hosp.plot(markersize=50, color='red', edgecolor='black')
plt.title("Hospitais em Niterói (OpenStreetMap)")
plt.axis('off')
plt.show()

### 5.5 Boas práticas ao consumir APIs

```python
import requests
import time

def requisicao_segura(url, params=None, tentativas=3, espera=2):
    """Faz requisição com tratamento de erros e retentativas."""
    for i in range(tentativas):
        try:
            resposta = requests.get(url, params=params, timeout=10)
            resposta.raise_for_status()  # Lança erro em status 4xx/5xx
            return resposta.json()
        except requests.exceptions.HTTPError as e:
            print(f"Erro HTTP: {e}")
        except requests.exceptions.ConnectionError:
            print(f"Erro de conexão. Tentativa {i+1}/{tentativas}")
            time.sleep(espera)
        except requests.exceptions.Timeout:
            print("Timeout na requisição")
    return None

# Respeite os limites de requisições (rate limiting)!
# Adicione pausas entre chamadas em loops:
codigos = [3304557, 3550308, 4314902]  # Niterói, São Paulo, Porto Alegre
for codigo in codigos:
    dados = requisicao_segura(f"https://servicodados.ibge.gov.br/api/v1/localidades/municipios/{codigo}")
    print(dados['nome'])
    time.sleep(0.5)  # Pausa de 0.5s entre requisições
```

## 6. Como obter dados sem serviço de API

Nem sempre os dados que precisamos estão disponíveis em uma API. Nesses casos, recorremos a outras técnicas de coleta.

### 6.1 Download Direto de Arquivos

A forma mais simples: o servidor disponibiliza arquivos para download direto.

In [ ]:
import requests
import os

def baixar_arquivo(url, destino):
    """Faz download de arquivo com barra de progresso."""
    if os.path.exists(destino):
        print(f"Arquivo já existe: {destino}")
        return
    
    resposta = requests.get(url, stream=True)
    resposta.raise_for_status()
    
    total = int(resposta.headers.get('content-length', 0))
    baixado = 0
    
    with open(destino, 'wb') as f:
        for chunk in resposta.iter_content(chunk_size=8192):
            f.write(chunk)
            baixado += len(chunk)
            if total:
                pct = baixado / total * 100
                print(f"\rProgresso: {pct:.1f}%", end="")
    print(f"\nBaixado: {destino}")

# Exemplo: Focos de queimadas do INPE (CSV)
url_queimadas = (
    "https://dataserver-coids.inpe.br/queimadas/queimadas/"
    "focos/csv/mensal/Brasil/focos_mensal_br_202401.csv"
)
baixar_arquivo(url_queimadas, "focos_jan2024.csv")

df_focos = pd.read_csv("focos_jan2024.csv")
print(df_focos.head())
print(f"Colunas: {list(df_focos.columns)}")

### 6.2 Web Scraping

**Web Scraping** é a técnica de extrair dados diretamente de páginas HTML quando não existe API disponível.

> ⚠️ **Atenção ética e legal:**
> - Verifique o arquivo `robots.txt` do site (`https://site.com/robots.txt`)
> - Leia os Termos de Uso
> - Não sobrecarregue os servidores (use `time.sleep`)
> - Use preferencialmente APIs quando disponíveis

#### Bibliotecas principais

| Biblioteca | Uso |
|-----------|-----|
| `requests` | Buscar o HTML da página |
| `BeautifulSoup` | Parsear e navegar o HTML |
| `selenium` | Controlar navegador (páginas com JavaScript) |
| `scrapy` | Framework completo para scraping em escala |

In [ ]:
# !pip install beautifulsoup4 lxml

import requests
from bs4 import BeautifulSoup
import pandas as pd

# Exemplo: Extrair tabela de municípios do RJ da Wikipedia
url = "https://pt.wikipedia.org/wiki/Lista_de_munic%C3%ADpios_do_Rio_de_Janeiro_por_popula%C3%A7%C3%A3o"

# Headers simulando um navegador real (boa prática)
headers = {
    'User-Agent': 'Mozilla/5.0 (compatible; DataScienceClass/1.0; educational purpose)'
}

resposta = requests.get(url, headers=headers)
resposta.raise_for_status()

# Parsear o HTML
soup = BeautifulSoup(resposta.text, 'lxml')

# Encontrar a tabela principal
tabela = soup.find('table', {'class': 'wikitable'})

# Extrair linhas
linhas = []
for tr in tabela.find_all('tr')[1:]:  # Pula o cabeçalho
    colunas = [td.get_text(strip=True) for td in tr.find_all(['td', 'th'])]
    if colunas:
        linhas.append(colunas)

df_wiki = pd.DataFrame(linhas)
print(df_wiki.head(10))

### 6.3 Scraping de Páginas com JavaScript (Selenium)

Muitos sites modernos carregam dados via JavaScript. O `requests` comum não consegue acessar esse conteúdo — precisamos de um **navegador headless**.

In [ ]:
# !pip install selenium webdriver-manager

# Exemplo ilustrativo de uso do Selenium
# (requer instalação do ChromeDriver)

"""
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# Configurando o navegador em modo headless (sem interface gráfica)
opcoes = Options()
opcoes.add_argument('--headless')
opcoes.add_argument('--no-sandbox')
opcoes.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(options=opcoes)

try:
    driver.get("https://www.exemplo-com-js.gov.br/consulta")
    
    # Espera o elemento aparecer (aguarda JS carregar)
    wait = WebDriverWait(driver, 10)
    tabela = wait.until(EC.presence_of_element_located((By.ID, "resultados")))
    
    # Agora extrai o HTML já renderizado
    html_renderizado = driver.page_source
    soup = BeautifulSoup(html_renderizado, 'lxml')
    # ... continua o scraping normalmente

finally:
    driver.quit()  # Sempre fechar o navegador!
"""
print("Código Selenium disponível para referência (comentado).")

### 6.4 Leitura Direta de Tabelas HTML com Pandas

Para tabelas HTML simples, o `pandas` oferece um atalho poderoso:

In [ ]:
import pandas as pd

# pd.read_html() retorna uma lista com TODAS as tabelas da página
url = "https://pt.wikipedia.org/wiki/Lista_de_munic%C3%ADpios_do_Rio_de_Janeiro_por_popula%C3%A7%C3%A3o"

tabelas = pd.read_html(url, encoding='utf-8')
print(f"Número de tabelas encontradas: {len(tabelas)}")

# Pegar a primeira tabela relevante
df = tabelas[0]
print(df.head(10))

## Complemento

### Resumo das Técnicas de Prospecção

| Técnica | Quando usar | Dificuldade | Exemplo |
|---------|-------------|-------------|----------|
| **API REST** | Dado disponível com interface padronizada | ⭐⭐ | IBGE, OpenWeather, OSM |
| **Download direto** | Arquivos estáticos disponíveis publicamente | ⭐ | IBGE geoftp, INPE |
| **pd.read_html** | Tabelas HTML simples e estáticas | ⭐ | Wikipedia |
| **BeautifulSoup** | Páginas HTML sem JavaScript | ⭐⭐⭐ | Portais governamentais |
| **Selenium** | Páginas que carregam dados via JS | ⭐⭐⭐⭐ | Sistemas dinâmicos |

### Ferramentas extras para dados geoespaciais

```python
# osmnx: dados de redes viárias e POIs do OpenStreetMap
import osmnx as ox
G = ox.graph_from_place("Niterói, Brasil", network_type="drive")
ox.plot_graph(G)

# censobr: dados do Censo IBGE direto em Python
from censobr import read_population
df = read_population(year=2022, simplified=True)

# geobr: malhas e recortes territoriais do Brasil
import geobr
muni = geobr.read_municipality(code_muni="RJ", year=2020)
```

## Lista de exercícios complementares

**Exercício 1.** Acesse a API do IBGE (`servicodados.ibge.gov.br`) e obtenha a lista de todos os municípios do estado do Rio de Janeiro. Quantos são?

**Exercício 2.** Utilizando a API SIDRA do IBGE (tabela 9514), obtenha a população de todos os municípios fluminenses no Censo 2022. Qual o município mais populoso? E o menos populoso?

**Exercício 3.** Faça o download da malha municipal do estado do RJ via `geoftp.ibge.gov.br`. Plote um mapa colorindo os municípios por população (use os dados do Exercício 2).

**Exercício 4.** Usando a Overpass API, obtenha a localização de todas as escolas (`amenity=school`) da cidade de Niterói. Quantas foram encontradas? Plote-as em um mapa.

**Exercício 5.** Utilize `pd.read_html()` para extrair a tabela de municípios do Rio de Janeiro da Wikipedia e compare com os dados obtidos via API do IBGE. Há diferenças? Quais?

**Exercício 6.** Crie uma função Python que, dado o nome de uma cidade brasileira, retorne um dicionário com: código IBGE, população (Censo 2022), área territorial e UF. Use a API do IBGE.

## Lista de exercícios suplementares

**Exercício S1.** Utilizando a API do INMET (`apitempo.inmet.gov.br`), obtenha os dados de precipitação acumulada das estações automáticas do estado do RJ para os últimos 30 dias. Qual estação registrou maior precipitação?

**Exercício S2.** Acesse o portal de dados abertos do DNIT e faça o download do shapefile de rodovias federais do Brasil. Calcule a extensão total (km) das rodovias federais que passam pelo estado do Rio de Janeiro.

**Exercício S3.** Usando `osmnx`, obtenha a rede viária (modo `drive`) de um bairro de sua escolha. Calcule: número de nós, número de arestas e extensão total da malha em km.

**Exercício S4 (desafio).** Construa um pipeline automatizado que:
1. Baixa dados populacionais dos municípios do RJ (API IBGE)
2. Baixa a malha municipal (IBGE geoftp)
3. Faz o join espacial entre geometria e população
4. Gera um mapa coroplético interativo com `folium`
5. Salva o resultado em um arquivo HTML

**Exercício S5 (desafio).** Implemente um web scraper ético que colete dados de uma tabela de um portal público de sua escolha (ex.: dados de licitações, notas fiscais ambientais, registros de queimadas). Inclua: tratamento de erros, respeito ao `robots.txt`, e pausa entre requisições.

<!-- **Exercicio 6 gabarito:**

```python
import numpy as np

a = np.array([3.0, 4.25, 5.5, 8.0])
l = np.array([4.5, 4.25, 5.5, 5.5])

# Cálculo do coeficiente angular (m) e intercepto (b)
A = np.vstack([a, np.ones(len(a))]).T

# X = [m, b]
X_a = np.linalg.inv(A.T @ A) @ A.T @ l
X_a, np.linalg.lstsq(A, l)[0]
``` -->

<!-- **Exercicio 7 gabarito:** (mantido original) -->

<!-- **Exercicio 14 gabarito** (mantido original) -->

<!-- **Exercício 15 gabarito** (mantido original) -->